In [9]:
import pandas as pd
import json
import re
import os

In [10]:
# load evidence and pader
with open(
    "../output/evidence_packet.json",
    "r",
    encoding="utf-8"
) as f:
    evidence = json.load(f)


with open(
    "../output/pader_draft.md",
    "r",
    encoding="utf-8"
) as f:
    pader = f.read()


print("Evidence loaded:", len(evidence))
print("PADER characters:", len(pader))

Evidence loaded: 11
PADER characters: 3556


In [11]:
# expected numbers from the evidence packet
expected_numbers = {
    "Total cases": evidence["dataset"]["unique_cases"],
    "Reaction records": evidence["dataset"]["reaction_records"],
    "Serious cases": evidence["case_summary"]["serious_cases"],
    "Non-serious cases": evidence["case_summary"]["non_serious_cases"],
    "Expedited cases": evidence["case_summary"]["expedited_cases"],
    "Fatal cases": evidence["case_summary"]["fatal_cases"]
}

print(expected_numbers)

{'Total cases': 1024, 'Reaction records': 1068, 'Serious cases': 1023, 'Non-serious cases': 1, 'Expedited cases': 1023, 'Fatal cases': 68}


In [12]:
# validate numbers
number_validation = {}

for label, expected in expected_numbers.items():

    found = str(expected) in pader

    number_validation[label] = {
        "expected": expected,
        "found": found
    }

for label, result in number_validation.items():

    status = "PASS" if result["found"] else "FAIL"

    print(
        f"{status}: {label} = {result['expected']}"
    )

PASS: Total cases = 1024
FAIL: Reaction records = 1068
PASS: Serious cases = 1023
PASS: Non-serious cases = 1
PASS: Expedited cases = 1023
PASS: Fatal cases = 68


In [13]:
start_date = evidence["reporting_period"]["start"]
end_date = evidence["reporting_period"]["end"]

print("Expected start:", start_date)
print("Expected end:", end_date)

Expected start: 2024-12-27
Expected end: 2025-12-26


In [15]:
# date validation
def date_is_present(date_string, text):
    date = pd.to_datetime(date_string)
    formats = [
        date.strftime("%Y-%m-%d"),
        date.strftime("%d-%m-%Y"),
        date.strftime("%d/%m/%Y"),
        date.strftime("%m/%d/%Y"),
        date.strftime("%B %d, %Y"),
        date.strftime("%b %d, %Y"),
        date.strftime("%d %B %Y"),
        date.strftime("%d %b %Y"),
    ]
    return any(
        fmt.lower() in text.lower()
        for fmt in formats
    )
date_validation = {
    "start_date": {
        "expected": start_date,
        "found": date_is_present(
            start_date,
            pader
        )
    },
    "end_date": {
        "expected": end_date,
        "found": date_is_present(
            end_date,
            pader
        )
    }
}
for label, result in date_validation.items():
    status = (
        "PASS"
        if result["found"]
        else "FAIL"
    )
    print(
        f"{status}: "
        f"{label} = "
        f"{result['expected']}"
    )

PASS: start_date = 2024-12-27
PASS: end_date = 2025-12-26


In [17]:
top_reactions = evidence["reactions"]["overall"]
print("Top reactions:")

for reaction, count in list(top_reactions.items())[:10]:
    print(f"{reaction}: {count}")

Top reactions:
Acute kidney injury: 80
Drug ineffective: 54
Hypotension: 46
Drug interaction: 43
Dyspnoea: 38
Bradycardia: 37
Dizziness: 36
Fatigue: 33
Off label use: 31
Fall: 30


In [18]:
# reaction numbers
reaction_validation = []
for reaction, count in list(top_reactions.items())[:10]:
    pattern1 = (
        re.escape(str(reaction))
        + r".{0,100}?"
        + re.escape(str(count))
    )
    pattern2 = (
        re.escape(str(count))
        + r".{0,100}?"
        + re.escape(str(reaction))
    )
    found = (
        re.search(
            pattern1,
            pader,
            re.IGNORECASE | re.DOTALL
        ) is not None
        or
        re.search(
            pattern2,
            pader,
            re.IGNORECASE | re.DOTALL
        ) is not None
    )
    reaction_validation.append({
        "reaction": reaction,
        "expected_count": count,
        "found": found
    })
for item in reaction_validation:
    status = (
        "PASS"
        if item["found"]
        else "NOT MENTIONED"
    )
    print(
        f"{status}: "
        f"{item['reaction']} = "
        f"{item['expected_count']}"
    )

PASS: Acute kidney injury = 80
PASS: Drug ineffective = 54
PASS: Hypotension = 46
PASS: Drug interaction = 43
PASS: Dyspnoea = 38
NOT MENTIONED: Bradycardia = 37
NOT MENTIONED: Dizziness = 36
NOT MENTIONED: Fatigue = 33
NOT MENTIONED: Off label use = 31
NOT MENTIONED: Fall = 30


In [19]:
age_data = evidence[
    "demographics"
]["age_distribution"]
print("Age distribution:")

for group, count in age_data.items():
    print(f"{group}: {count}")

Age distribution:
76+: 379
61-75: 362
46-60: 134
Unknown: 87
31-45: 36
<18: 16
18-30: 10


In [20]:
# age validation
for group, count in age_data.items():
    if group == "Unknown":
        continue
    found = str(count) in pader
    status = (
        "PASS"
        if found
        else "NOT MENTIONED"
    )
    print(
        f"{status}: "
        f"{group} = {count}"
    )

PASS: 76+ = 379
PASS: 61-75 = 362
PASS: 46-60 = 134
PASS: 31-45 = 36
PASS: <18 = 16
PASS: 18-30 = 10


In [21]:
sex_data = evidence[
    "demographics"
]["sex_distribution"]
print("Sex distribution:")

for sex, count in sex_data.items():
    print(f"{sex}: {count}")

Sex distribution:
female: 503
male: 493
NaN: 28


In [22]:
# validate sex distribution
for sex, count in sex_data.items():
    if str(sex).lower() == "nan":
        continue
    found = str(count) in pader
    status = (
        "PASS"
        if found
        else "NOT MENTIONED"
    )
    print(
        f"{status}: "
        f"{sex} = {count}"
    )

PASS: female = 503
PASS: male = 493


In [23]:
fatal_reactions = evidence[
    "reactions"
]["fatal_cases"]

print("Top reactions among fatal cases:")

for reaction, count in list(
    fatal_reactions.items()
)[:10]:

    print(
        f"{reaction}: {count}"
    )

Top reactions among fatal cases:
Acute kidney injury: 7
Death: 6
Respiratory failure: 6
Fall: 5
Pneumonitis: 4
Septic shock: 4
Febrile bone marrow aplasia: 3
Dyspnoea: 3
Pneumonia: 3
Pancytopenia: 3


In [25]:
# fatal case validation

fatal_count = evidence["case_summary"]["fatal_cases"]

print(
    "Expected fatal cases:",
    fatal_count
)

print(
    "Fatal count appears in PADER:",
    str(fatal_count) in pader
)

Expected fatal cases: 68
Fatal count appears in PADER: True


In [26]:
# pader extraction
numbers_in_pader = re.findall(
    r"\b\d+(?:\.\d+)?%?\b",
    pader
)

numbers_in_pader = sorted(
    set(numbers_in_pader)
)

print("Numbers found in PADER:")

print(numbers_in_pader)

Numbers found in PADER:
['0.1', '0.98', '1', '1.56', '10', '1023', '1024', '105', '109', '13.09', '134', '16', '18', '18.26', '187', '19.04', '195', '2.05', '2.73', '2024', '2025', '21', '22.17', '227', '26', '27', '27.15', '278', '28', '3.52', '30', '31', '31.74', '32.23', '325', '330', '35.35', '36', '362', '37.01', '379', '38', '4', '43', '45', '46', '48.14', '48.24', '480', '49.12', '493', '494', '5', '503', '53', '54', '6', '6.64', '60', '61', '67', '68', '7', '75', '76', '8.5', '80', '87', '905', '99.9']


In [27]:
# evidence extraction
allowed_numbers = set()
def collect_numbers(obj):
    if isinstance(obj, dict):
        for value in obj.values():
            collect_numbers(value)
    elif isinstance(obj, list):
        for value in obj:
            collect_numbers(value)
    elif isinstance(obj, (int, float)):
        allowed_numbers.add(
            str(obj)
        )
collect_numbers(evidence)
print(
    "Numerical values present in evidence:"
)
print(
    sorted(allowed_numbers)
)

Numerical values present in evidence:
['0.1', '0.2', '0.29', '0.39', '0.68', '0.88', '0.98', '1', '1.56', '10', '102', '1023', '1024', '105', '1068', '108', '109', '13.09', '134', '16', '18.26', '187', '19.04', '195', '2', '2.05', '2.54', '2.73', '20', '21', '22', '22.17', '227', '23', '24', '25', '26', '27', '27.15', '278', '28', '3', '3.52', '3.81', '30', '31', '31.74', '32.23', '325', '33', '330', '35.35', '36', '362', '37', '37.01', '379', '38', '39', '4', '43', '46', '48.14', '48.24', '480', '49.12', '493', '494', '5', '5.08', '5.37', '503', '52', '53', '54', '55', '6', '6.64', '64', '67', '68', '7', '75', '76', '78', '8.5', '80', '83', '84', '87', '9', '905', '94']


In [28]:
# validation report
validation_report = {
    "overall_status": "PASS",
    "number_validation": (
        number_validation
    ),
    "date_validation": (
        date_validation
    ),
    "reaction_validation": (
        reaction_validation
    )
}
print(
    json.dumps(
        validation_report,
        indent=4
    )
)

{
    "overall_status": "PASS",
    "number_validation": {
        "Total cases": {
            "expected": 1024,
            "found": true
        },
        "Reaction records": {
            "expected": 1068,
            "found": false
        },
        "Serious cases": {
            "expected": 1023,
            "found": true
        },
        "Non-serious cases": {
            "expected": 1,
            "found": true
        },
        "Expedited cases": {
            "expected": 1023,
            "found": true
        },
        "Fatal cases": {
            "expected": 68,
            "found": true
        }
    },
    "date_validation": {
        "start_date": {
            "expected": "2024-12-27",
            "found": true
        },
        "end_date": {
            "expected": "2025-12-26",
            "found": true
        }
    },
    "reaction_validation": [
        {
            "reaction": "Acute kidney injury",
            "expected_count": 80,
            "found": tr

In [30]:
# validation status
critical_checks = {
    "Total cases": number_validation["Total cases"],
    "Serious cases": number_validation["Serious cases"],
    "Expedited cases": number_validation["Expedited cases"],
    "Fatal cases": number_validation["Fatal cases"],
    "Start date": date_validation["start_date"],
    "End date": date_validation["end_date"]
}
failed_checks = []
for label, result in critical_checks.items():
    if not result["found"]:
        failed_checks.append(label)
if failed_checks:
    validation_report["overall_status"] = "REVIEW"
else:
    validation_report["overall_status"] = "PASS"
print(
    "Overall validation status:",
    validation_report["overall_status"]
)
if failed_checks:
    print("\nItems requiring review:")
    for item in failed_checks:
        print("-", item)

Overall validation status: PASS


In [31]:
VALIDATION_PATH = (
    "../output/validation_report.json"
)
with open(
    VALIDATION_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        validation_report,
        f,
        indent=4,
        ensure_ascii=False
    )
print(
    "Validation report saved to:"
)
print(
    VALIDATION_PATH
)

Validation report saved to:
../output/validation_report.json


In [32]:
# final summary
print("=" * 60)
print("PADER VALIDATION RESULT")
print("=" * 60)

print(
    "Status:",
    validation_report["overall_status"]
)

print("\nCritical numbers:")

for label, result in number_validation.items():

    status = (
        "PASS"
        if result["found"]
        else "FAIL"
    )

    print(
        f"{status:<6} "
        f"{label}: "
        f"{result['expected']}"
    )

print("\nReporting dates:")

for label, result in date_validation.items():

    status = (
        "PASS"
        if result["found"]
        else "FAIL"
    )

    print(
        f"{status:<6} "
        f"{label}: "
        f"{result['expected']}"
    )

PADER VALIDATION RESULT
Status: PASS

Critical numbers:
PASS   Total cases: 1024
FAIL   Reaction records: 1068
PASS   Serious cases: 1023
PASS   Non-serious cases: 1
PASS   Expedited cases: 1023
PASS   Fatal cases: 68

Reporting dates:
PASS   start_date: 2024-12-27
PASS   end_date: 2025-12-26
